# Terminal demand / supply — PyGWalker explorer

Interactive Tableau-style drag-and-drop on a **terminal profile** built from
`data/derived/tableau/trades_imp_exp_master`.

**Goal:** classify terminals by import vs export behavior and product mix, so we
can read demand vs supply character (start filtering to **Spain** in the UI).

**Defaults**
- Scope: global (filter Country in Graphic Walker)
- Grain: one row per `Installation`
- Status: `Trade status = Delivered`

**Units:** volumes in **kb** = `Volume (bbl) / 1000` (same basis as `kb_imp_exp`,
but we use absolute kb so import and export are both positive).

---

### What is PyGWalker?

PyGWalker embeds [Graphic Walker](https://github.com/Kanaries/graphic-walker)
inside the notebook. You get a field list (dimensions / measures) and a chart
canvas — drag fields to **X / Y / Color / Size / Filters**, same mental model as
Tableau.

We pass an already-aggregated profile (not 1.5M event rows) so the UI stays fast.
`kernel_computation=True` runs aggregations in Python/DuckDB instead of the browser.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
import pygwalker as pyg

ROOT = Path("..").resolve()
TABLEAU_DIR = ROOT / "data" / "derived" / "tableau"
PARQUET_PATH = TABLEAU_DIR / "trades_imp_exp_master.parquet"
CSV_PATH = TABLEAU_DIR / "trades_imp_exp_master.csv"

# Keep the profile focused on terminals that actually moved product.
STATUS = "Delivered"
MIN_TOTAL_KB = 1.0  # drop near-empty noise terminals from the explorer

# Behavior buckets — tweak after you inspect Spain in the UI.
IMPORT_HEAVY_MIN = 0.70
EXPORT_HEAVY_MAX = 0.30

print("pygwalker", getattr(pyg, "__version__", "?"))
print("ROOT", ROOT)

pygwalker 0.5.0.1
ROOT C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\prod_nowcast


c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\.venv\Lib\site-packages\requests\__init__.py:109: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


## 1) Load delivered trades

Prefer parquet (faster). The master table already stacks each trade as Export
(origin location) and Import (destination location) — see `build_tableau_imp_exp.py`.

For a terminal row we therefore treat:
- **Export kb** → product leaving that installation (supply / origin)
- **Import kb** → product arriving at that installation (demand / sink)

In [2]:
USE_COLS = [
    "Imp/Exp",
    "kb_imp_exp",
    "Installation",
    "Country",
    "Zone",
    "Continent",
    "Subcontinent",
    "TradingRegion",
    "Date",
    "Family",
    "Product",
    "Volume (bbl)",
    "Trade status",
]

if PARQUET_PATH.exists():
    raw = pd.read_parquet(PARQUET_PATH, columns=USE_COLS)
    source = PARQUET_PATH.name
else:
    assert CSV_PATH.exists(), f"Missing master table under {TABLEAU_DIR}"
    raw = pd.read_csv(CSV_PATH, usecols=USE_COLS, parse_dates=["Date"])
    source = CSV_PATH.name

# Explicit status filter even if the current extract is already Delivered-only.
events = raw.loc[raw["Trade status"].eq(STATUS)].copy()
events = events.loc[events["Installation"].notna() & events["Installation"].ne("")]

# Absolute kb so import/export can be compared as positive volumes.
events["kb_abs"] = events["Volume (bbl)"] / 1000.0

print(f"loaded {source}: {len(raw):,} rows → {len(events):,} after {STATUS} + Installation")
print("date range:", events["Date"].min(), "→", events["Date"].max())
print("countries:", events["Country"].nunique(), "| terminals:", events["Installation"].nunique())
print("Spain rows:", int(events["Country"].eq("Spain").sum()))

loaded trades_imp_exp_master.parquet: 1,545,966 rows → 1,406,959 after Delivered + Installation
date range: 2016-01-02 19:09:00 → 2026-07-16 09:59:00
countries: 176 | terminals: 3067
Spain rows: 42852


## 2) Build terminal profile (one row per Installation)

Metrics designed for typology:
- `import_share` — fraction of throughput that is inbound
- `net_kb` — `import_kb - export_kb` (positive ≈ net demand sink)
- `top_family` / `top_family_share` — dominant product family concentration
- `behavior` — coarse label from import share thresholds

In [3]:
def _mode_or_first(s: pd.Series):
    """Stable geo label when a terminal rarely appears under mixed attributes."""
    m = s.dropna()
    if m.empty:
        return pd.NA
    modes = m.mode()
    return modes.iloc[0] if len(modes) else m.iloc[0]


flows = (
    events.groupby(["Installation", "Imp/Exp"], dropna=False)["kb_abs"]
    .sum()
    .unstack(fill_value=0.0)
)
for col in ("Import", "Export"):
    if col not in flows.columns:
        flows[col] = 0.0

flows = flows.rename(columns={"Import": "import_kb", "Export": "export_kb"})
flows["total_kb"] = flows["import_kb"] + flows["export_kb"]
flows["net_kb"] = flows["import_kb"] - flows["export_kb"]
flows["import_share"] = flows["import_kb"] / flows["total_kb"].replace(0, pd.NA)

trade_counts = (
    events.groupby(["Installation", "Imp/Exp"], dropna=False)
    .size()
    .unstack(fill_value=0)
    .rename(columns={"Import": "import_trades", "Export": "export_trades"})
)
for col in ("import_trades", "export_trades"):
    if col not in trade_counts.columns:
        trade_counts[col] = 0

geo = (
    events.groupby("Installation", dropna=False)
    .agg(
        Country=("Country", _mode_or_first),
        Zone=("Zone", _mode_or_first),
        Continent=("Continent", _mode_or_first),
        Subcontinent=("Subcontinent", _mode_or_first),
        TradingRegion=("TradingRegion", _mode_or_first),
        first_date=("Date", "min"),
        last_date=("Date", "max"),
        n_products=("Product", "nunique"),
        n_families=("Family", "nunique"),
    )
)

# Dominant family by absolute kb at the terminal (helps product-demand typing).
family_kb = (
    events.groupby(["Installation", "Family"], dropna=False)["kb_abs"]
    .sum()
    .reset_index()
)
family_kb["fam_rank"] = family_kb.groupby("Installation")["kb_abs"].rank(
    method="first", ascending=False
)
top_family = (
    family_kb.loc[family_kb["fam_rank"].eq(1), ["Installation", "Family", "kb_abs"]]
    .rename(columns={"Family": "top_family", "kb_abs": "top_family_kb"})
    .set_index("Installation")
)

product_kb = (
    events.groupby(["Installation", "Product"], dropna=False)["kb_abs"]
    .sum()
    .reset_index()
)
product_kb["prod_rank"] = product_kb.groupby("Installation")["kb_abs"].rank(
    method="first", ascending=False
)
top_product = (
    product_kb.loc[product_kb["prod_rank"].eq(1), ["Installation", "Product", "kb_abs"]]
    .rename(columns={"Product": "top_product", "kb_abs": "top_product_kb"})
    .set_index("Installation")
)

profile = (
    geo.join(flows[["import_kb", "export_kb", "total_kb", "net_kb", "import_share"]])
    .join(trade_counts[["import_trades", "export_trades"]])
    .join(top_family)
    .join(top_product)
    .reset_index()
)

profile["total_trades"] = profile["import_trades"] + profile["export_trades"]
profile["top_family_share"] = profile["top_family_kb"] / profile["total_kb"].replace(0, pd.NA)
profile["top_product_share"] = profile["top_product_kb"] / profile["total_kb"].replace(0, pd.NA)


def classify_behavior(share: float) -> str:
    if pd.isna(share):
        return "unknown"
    if share >= IMPORT_HEAVY_MIN:
        return "import_heavy"
    if share <= EXPORT_HEAVY_MAX:
        return "export_heavy"
    return "balanced"


profile["behavior"] = profile["import_share"].map(classify_behavior)
profile = profile.loc[profile["total_kb"] >= MIN_TOTAL_KB].copy()

# Friendlier dtypes for Graphic Walker field inference.
for c in ("first_date", "last_date"):
    profile[c] = pd.to_datetime(profile[c])

profile = profile.sort_values("total_kb", ascending=False).reset_index(drop=True)

print(f"terminal profile: {len(profile):,} installations")
print(profile["behavior"].value_counts())
print("\nSpain preview:")
display(
    profile.loc[profile["Country"].eq("Spain")]
    .head(12)[
        [
            "Installation",
            "Zone",
            "import_kb",
            "export_kb",
            "import_share",
            "behavior",
            "top_family",
            "top_product",
        ]
    ]
)

terminal profile: 3,067 installations
behavior
import_heavy    1852
export_heavy     736
balanced         479
Name: count, dtype: int64

Spain preview:


,Installation,Zone,import_kb,export_kb,import_share,behavior,top_family,top_product
76,Repsol Cartagena Refinery,Cartagena,98369.144,372825.098,0.208766,export_heavy,Light Ends,Naphtha
124,Gibraltar-San Roque Refinery,Algeciras,94866.524,241113.279,0.282358,export_heavy,Middle Distillates,Gasoline
126,Bilbao Oil Facilities,Bilbao,109649.497,225512.999,0.327153,balanced,Middle Distillates,Gasoline
148,Tarragona Refinery,Tarragona,138391.799,149822.575,0.480170,balanced,Light Ends,Naphtha
181,Vopak Algeciras,Algeciras,204171.281,36655.465,0.847793,import_heavy,DPP,FO
184,Vopak Terquimsa Barcelona,Barcelona,179852.107,58907.576,0.753277,import_heavy,Middle Distillates,Diesel
205,Torre Arenillas,Huelva,7917.976,206556.635,0.036918,export_heavy,Middle Distillates,Naphtha
212,Repsol Coruna Refinery,Coruna,52150.621,157109.708,0.249214,export_heavy,Light Ends,Gasoil
256,BP Castellon Refinery,Castellon,37158.514,136772.321,0.213640,export_heavy,Middle Distillates,Gasoil
280,Oryx Las Palmas,Las Palmas,113545.335,43355.150,0.723677,import_heavy,Middle Distillates,Gasoil


## 3) Drag-and-drop explorer

### How to use Graphic Walker (first 5 minutes)

1. Run the cell below — the UI appears under it.
2. **Filter** → drag `Country` → choose **Spain** (or type-ahead).
3. Try these starter charts:
   - **Bar:** X = `Installation`, Y = `total_kb` (Mark: bar). Sort descending.
   - **Import share:** X = `Installation`, Y = `import_share`, Color = `behavior`.
   - **Supply vs demand scatter:** X = `export_kb`, Y = `import_kb`, Color = `top_family`, Size = `total_kb`.
   - **Product character:** X = `top_product`, Y = `total_kb`, Color = `behavior`.
4. Use **+** to add another chart to the same dashboard.
5. Optional: export the chart spec / save config later via `spec=` if you want to persist layouts.

### How to read `behavior`

| Label | Rule (default) | Interpretation |
|---|---|---|
| `import_heavy` | import_share ≥ 0.70 | Demand sink / consuming terminal |
| `export_heavy` | import_share ≤ 0.30 | Supply source / loading terminal |
| `balanced` | in between | Hub / dual-role / blending |

Thresholds are in the setup cell — change them after you eyeball Spain.

In [4]:
# kernel_computation keeps heavy field ops out of the browser (safer as we grow).
walker = pyg.walk(
    raw,
    kernel_computation=True,
    default_tab="vis",
)
walker

Box(children=(HTML(value='\n<div id="ifr-pyg-0006571bc82001daED7OxFJoXtYPuWKH" style="height: auto">\n    <hea…

## 4) Optional: Spain table (pandas sanity check)

Use this if you want a static ranked list beside the drag-and-drop UI.

In [5]:
spain = profile.loc[profile["Country"].eq("Spain")].copy()
print(f"Spain terminals: {len(spain)}")
print(spain["behavior"].value_counts())
spain[
    [
        "Installation",
        "Zone",
        "TradingRegion",
        "total_kb",
        "import_kb",
        "export_kb",
        "import_share",
        "net_kb",
        "behavior",
        "top_family",
        "top_family_share",
        "top_product",
        "n_products",
    ]
].head(25)

Spain terminals: 69
behavior
import_heavy    38
balanced        16
export_heavy    15
Name: count, dtype: int64


,Installation,Zone,TradingRegion,total_kb,import_kb,export_kb,import_share,net_kb,behavior,top_family,top_family_share,top_product,n_products
76,Repsol Cartagena Refinery,Cartagena,MED Zone (MED Sea+Black Sea),471194.242,98369.144,372825.098,0.208766,-274455.954,export_heavy,Light Ends,0.405636,Naphtha,16
124,Gibraltar-San Roque Refinery,Algeciras,MED Zone (MED Sea+Black Sea),335979.803,94866.524,241113.279,0.282358,-146246.755,export_heavy,Middle Distillates,0.489997,Gasoline,16
126,Bilbao Oil Facilities,Bilbao,North West Europe Zone,335162.496,109649.497,225512.999,0.327153,-115863.502,balanced,Middle Distillates,0.463700,Gasoline,13
148,Tarragona Refinery,Tarragona,MED Zone (MED Sea+Black Sea),288214.374,138391.799,149822.575,0.480170,-11430.776,balanced,Light Ends,0.490009,Naphtha,13
181,Vopak Algeciras,Algeciras,MED Zone (MED Sea+Black Sea),240826.746,204171.281,36655.465,0.847793,167515.816,import_heavy,DPP,0.884556,FO,14
184,Vopak Terquimsa Barcelona,Barcelona,MED Zone (MED Sea+Black Sea),238759.683,179852.107,58907.576,0.753277,120944.531,import_heavy,Middle Distillates,0.564485,Diesel,13
205,Torre Arenillas,Huelva,MED Zone (MED Sea+Black Sea),214474.611,7917.976,206556.635,0.036918,-198638.659,export_heavy,Middle Distillates,0.510785,Naphtha,9
212,Repsol Coruna Refinery,Coruna,North West Europe Zone,209260.329,52150.621,157109.708,0.249214,-104959.087,export_heavy,Light Ends,0.393055,Gasoil,12
256,BP Castellon Refinery,Castellon,MED Zone (MED Sea+Black Sea),173930.835,37158.514,136772.321,0.213640,-99613.807,export_heavy,Middle Distillates,0.492189,Gasoil,10
280,Oryx Las Palmas,Las Palmas,<NA>,156900.485,113545.335,43355.150,0.723677,70190.185,import_heavy,Middle Distillates,0.767931,Gasoil,13
